# ResNet101 - KHOTAA Diabetic Foot Ulcer Classification

## 1. Imports & Configuration

In [5]:
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import resnet101, ResNet101_Weights
import numpy as np
from sklearn.model_selection import StratifiedKFold
import importlib

sys.path.append('../')
sys.path.append('./')

from dataset_loader import SplitFolderDatasetLoader
from dataset_preprocessing import DFUPreprocessing
from utils import checkpoint_manager, training_engine
importlib.reload(checkpoint_manager)
importlib.reload(training_engine)
from utils.checkpoint_manager import CheckpointManager
from utils.training_engine import TrainingEngine, create_optimizer
from utils.metrics_evaluator import (
    calculate_metrics, print_metrics, plot_confusion_matrix,
    plot_roc_curve, plot_training_history
)

print("✓ Imports complete (modules reloaded)")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

✓ Imports complete (modules reloaded)
PyTorch version: 2.8.0
CUDA available: False


## 2. Load Dataset

In [6]:
# Load dataset
loader = SplitFolderDatasetLoader(root_dir='../../dfu-dataset-annotated-into-4-classes')
classes = loader.get_classes()
num_classes = loader.get_num_classes()

print(f"Classes: {classes}")
print(f"Number of classes: {num_classes}")

# Initialize preprocessing
preprocessor = DFUPreprocessing()
train_transform = preprocessor.get_train_transforms()
val_test_transform = preprocessor.get_valid_test_transforms()

# Dataset class
class DFUDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        from PIL import Image
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

# Prepare data for cross-validation
X_train, y_train = loader.load_split_paths('train', shuffle=True)
X_val, y_val = loader.load_split_paths('valid')
X_all = np.concatenate([X_train, X_val])
y_all = np.concatenate([y_train, y_val])

# Test set (untouched until final evaluation)
X_test, y_test = loader.load_split_paths('test')
test_dataset = DFUDataset(X_test, y_test, transform=val_test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# Initialize 5-fold stratified cross-validation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"\nTotal training samples (train+valid): {len(X_all)}")
print(f"Test samples: {len(X_test)}")
print("✓ Dataset loaded and ready for 5-fold cross-validation")

[DatasetLoader] Root: /Users/manaralharbi/Desktop/KHOTAA/dfu-dataset-annotated-into-4-classes
[DatasetLoader] Splits: ['train', 'valid', 'test']
[DatasetLoader] Classes (4): ['Grade 1', 'Grade 2', 'Grade 3', 'Grade 4']
Classes: ['Grade 1', 'Grade 2', 'Grade 3', 'Grade 4']
Number of classes: 4
[DFUPreprocessing] Initialized
[DFUPreprocessing] Image size: 224x224
[DFUPreprocessing] Train: with augmentation
[DFUPreprocessing] Valid/Test: no augmentation
[DatasetLoader] Split 'train': 9639 images
[DatasetLoader] Split 'valid': 282 images
[DatasetLoader] Split 'test': 141 images

Total training samples (train+valid): 9921
Test samples: 141
✓ Dataset loaded and ready for 5-fold cross-validation


## 3. Model Definition

In [7]:
# Setup device and loss function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.CrossEntropyLoss()

print(f"Device: {device}")

# Create ResNet101 model
def create_resnet101_model(num_classes=4, pretrained=True):
    """
    Create ResNet101 model for DFU classification.
    
    Args:
        num_classes: Number of output classes (4 for DFU grades)
        pretrained: Use ImageNet pretrained weights
    
    Returns:
        ResNet101 model configured for DFU classification
    """
    if pretrained:
        model = models.resnet101(weights=ResNet101_Weights.IMAGENET1K_V2)
    else:
        model = models.resnet101(weights=None)
    
    # Modify final fully connected layer
    # ResNet101 fc layer: Linear(2048 -> 1000)
    # Replace with: Linear(2048 -> num_classes)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    
    return model

# Test model creation
test_model = create_resnet101_model(num_classes=num_classes)
print(f"\n✓ ResNet101 model created")
print(f"Input size: 224x224")
print(f"Output classes: {num_classes}")
print(f"Final FC layer: {test_model.fc}")

Device: cpu

✓ ResNet101 model created
Input size: 224x224
Output classes: 4
Final FC layer: Linear(in_features=2048, out_features=4, bias=True)


## 4. Training

## 3.1 Learning Rate Schedule Explanation

### Current Configuration:
- **Initial Learning Rate**: `0.001`
- **Scheduler**: `StepLR(step_size=10, gamma=0.1)`
- **Maximum Epochs**: `30`

### How Learning Rate Changes:

The `StepLR` scheduler reduces the learning rate by multiplying it by `gamma` every `step_size` epochs:

| Epoch Range | Learning Rate | Calculation |
|-------------|---------------|-------------|
| 1-10        | 0.001         | Initial LR |
| 11-20       | 0.0001        | 0.001 × 0.1 |
| 21-30       | 0.00001       | 0.0001 × 0.1 |

### Why This Schedule?

1. **Epochs 1-10 (LR = 0.001)**: 
   - Fast initial learning with larger weight updates
   - Model quickly learns broad patterns and features
   - Training loss decreases rapidly

2. **Epochs 11-20 (LR = 0.0001)**: 
   - Slower, more refined learning
   - Fine-tunes the features learned in first 10 epochs
   - Helps avoid overshooting optimal weights

3. **Epochs 21-30 (LR = 0.00001)**: 
   - Very fine-grained adjustments
   - Polishes model weights for best performance
   - Minimizes risk of divergence

**Note**: With early stopping (patience=7), training may stop before reaching all 30 epochs if validation accuracy doesn't improve.


In [ ]:
# 5-Fold Cross-Validation Training
fold_results = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_all, y_all), 1):
    print(f"\n{'='*60}\nFOLD {fold}/5\n{'='*60}")
    
    # Prepare fold data
    X_train_fold = X_all[train_idx]
    y_train_fold = y_all[train_idx]
    X_val_fold = X_all[val_idx]
    y_val_fold = y_all[val_idx]
    
    train_dataset = DFUDataset(X_train_fold, y_train_fold, transform=train_transform)
    val_dataset = DFUDataset(X_val_fold, y_val_fold, transform=val_test_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
    
    # Create model
    model = create_resnet101_model(num_classes=num_classes, pretrained=True)
    model = model.to(device)
    
    # Setup optimizer and scheduler
    optimizer = create_optimizer(model, lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
    checkpoint_manager = CheckpointManager(base_dir='checkpoints', experiment_name=f'resnet101_fold{fold}')
    engine = TrainingEngine(model=model, device=device)
    
    # Train
    history = engine.train(
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        num_epochs=30,
        scheduler=scheduler,
        checkpoint_manager=checkpoint_manager,
        early_stopping_patience=7,
        use_early_stopping=True,
        verbose=True
    )
    
    # Store results
    best_val_acc = max(history['val_acc'])
    fold_results.append({
        'fold': fold,
        'best_val_acc': best_val_acc,
        'final_val_acc': history['val_acc'][-1],
        'stopped_epoch': history['stopped_epoch'],
        'history': history
    })
    print(f"Fold {fold} - Best Acc: {best_val_acc*100:.2f}% (stopped at epoch {history['stopped_epoch']})")

# Cross-validation summary
avg_acc = np.mean([r['best_val_acc'] for r in fold_results])
std_acc = np.std([r['best_val_acc'] for r in fold_results])
avg_epochs = np.mean([r['stopped_epoch'] for r in fold_results])

print(f"\n{'='*60}")
print(f"5-FOLD CROSS-VALIDATION RESULTS")
print(f"{'='*60}")
print(f"Mean Accuracy: {avg_acc*100:.2f}% ± {std_acc*100:.2f}%")
print(f"Average Epochs: {avg_epochs:.1f}")
print(f"\nIndividual Fold Results:")
for r in fold_results:
    print(f"  Fold {r['fold']}: {r['best_val_acc']*100:.2f}% (epoch {r['stopped_epoch']})")
print(f"{'='*60}")


FOLD 1/5
Created SGD optimizer: lr=0.001, momentum=0.8, weight_decay=0.0001
Training on: cpu

Epoch 1/30


Evaluating: 100%|██████████| 63/63 [06:55<00:00,  6.60s/it, loss=1.1505, acc=53.30%]


Learning Rate: 0.001000

Epoch 1 Results:
   Train Loss: 1.2725 | Train Acc: 43.36%
   Val Loss:   1.1742 | Val Acc:   53.30%
SUCCESS: Checkpoint saved: checkpoints/resnet101_fold1/fold_1/checkpoint_epoch_1.pth
SUCCESS: Best model saved: checkpoints/resnet101_fold1/fold_1/best_accuracy.pt
New best model! Val Acc: 53.30%

Epoch 2/30


Evaluating: 100%|██████████| 63/63 [06:45<00:00,  6.44s/it, loss=0.7342, acc=61.81%]


Learning Rate: 0.001000

Epoch 2 Results:
   Train Loss: 1.0646 | Train Acc: 56.24%
   Val Loss:   0.9748 | Val Acc:   61.81%
SUCCESS: Checkpoint saved: checkpoints/resnet101_fold1/fold_1/checkpoint_epoch_2.pth
SUCCESS: Best model saved: checkpoints/resnet101_fold1/fold_1/best_accuracy.pt
New best model! Val Acc: 61.81%

Epoch 3/30


...

Evaluating: 100%|██████████| 62/62 [07:33<00:00,  7.32s/it, loss=0.7897, acc=91.18%]


Learning Rate: 0.000010

Epoch 27 Results:
   Train Loss: 0.2981 | Train Acc: 89.63%
   Val Loss:   0.2855 | Val Acc:   91.18%
SUCCESS: Checkpoint saved: checkpoints/resnet101_fold4/fold_1/checkpoint_epoch_27.pth

Epoch 28/30


Evaluating: 100%|██████████| 62/62 [07:32<00:00,  7.30s/it, loss=0.7696, acc=91.08%]


Learning Rate: 0.000010

Epoch 28 Results:
   Train Loss: 0.2993 | Train Acc: 89.50%
   Val Loss:   0.2848 | Val Acc:   91.08%
SUCCESS: Checkpoint saved: checkpoints/resnet101_fold4/fold_1/checkpoint_epoch_28.pth

Epoch 29/30


Training:  15%|█▌        | 38/249 [06:11<31:36,  8.99s/it, loss=0.3188, acc=90.38%]

## 4.1 Resume Training (if interrupted)

Run this cell if training was interrupted. It will:
- Check which folds have been completed
- Continue training from the next fold
- Preserve all previous fold results


In [ ]:
# Resume Training from Last Checkpoint
import os
import json

print("="*60)
print("RESUMING TRAINING FROM CHECKPOINT")
print("="*60)

# Check which fold was interrupted
print(f"\nFold results collected so far: {len(fold_results)}/5")

# Determine which fold to resume from
completed_folds = len(fold_results)
next_fold_to_train = completed_folds + 1

print(f"Completed folds: {completed_folds}")
print(f"Next fold to train: {next_fold_to_train}")

if next_fold_to_train <= 5:
    print(f"\n{'='*60}")
    print(f"CONTINUING TRAINING FROM FOLD {next_fold_to_train}")
    print(f"{'='*60}")
    
    # Continue with remaining folds
    for fold_num, (train_idx, val_idx) in enumerate(kfold.split(X_all, y_all), 1):
        if fold_num < next_fold_to_train:
            continue  # Skip already completed folds
        
        print(f"\n{'='*60}\nFOLD {fold_num}/5\n{'='*60}")
        
        # Prepare fold data
        X_train_fold = X_all[train_idx]
        y_train_fold = y_all[train_idx]
        X_val_fold = X_all[val_idx]
        y_val_fold = y_all[val_idx]
        
        train_dataset = DFUDataset(X_train_fold, y_train_fold, transform=train_transform)
        val_dataset = DFUDataset(X_val_fold, y_val_fold, transform=val_test_transform)
        
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
        
        # Create model
        model = create_resnet101_model(num_classes=num_classes, pretrained=True)
        model = model.to(device)
        
        # Setup optimizer and scheduler
        optimizer = create_optimizer(model, lr=0.001)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
        checkpoint_manager = CheckpointManager(base_dir='checkpoints', experiment_name=f'resnet101_fold{fold_num}')
        engine = TrainingEngine(model=model, device=device)
        
        # Train
        history = engine.train(
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            num_epochs=30,
            scheduler=scheduler,
            checkpoint_manager=checkpoint_manager,
            early_stopping_patience=7,
            use_early_stopping=True,
            verbose=True
        )
        
        # Store results
        best_val_acc = max(history['val_acc'])
        fold_results.append({
            'fold': fold_num,
            'best_val_acc': best_val_acc,
            'final_val_acc': history['val_acc'][-1],
            'stopped_epoch': history['stopped_epoch'],
            'history': history
        })
        print(f"Fold {fold_num} - Best Acc: {best_val_acc*100:.2f}% (stopped at epoch {history['stopped_epoch']})")
    
    # Display final cross-validation summary
    avg_acc = np.mean([r['best_val_acc'] for r in fold_results])
    std_acc = np.std([r['best_val_acc'] for r in fold_results])
    avg_epochs = np.mean([r['stopped_epoch'] for r in fold_results])
    
    print(f"\n{'='*60}")
    print(f"5-FOLD CROSS-VALIDATION RESULTS (COMPLETE)")
    print(f"{'='*60}")
    print(f"Mean Accuracy: {avg_acc*100:.2f}% ± {std_acc*100:.2f}%")
    print(f"Average Epochs: {avg_epochs:.1f}")
    print(f"\nIndividual Fold Results:")
    for r in fold_results:
        print(f"  Fold {r['fold']}: {r['best_val_acc']*100:.2f}% (epoch {r['stopped_epoch']})")
    print(f"{'='*60}")
else:
    print("\n✓ All 5 folds have been completed!")
    print(f"\nYou can now proceed to the evaluation cell.")
    
    # Display summary
    avg_acc = np.mean([r['best_val_acc'] for r in fold_results])
    std_acc = np.std([r['best_val_acc'] for r in fold_results])
    avg_epochs = np.mean([r['stopped_epoch'] for r in fold_results])
    
    print(f"\n{'='*60}")
    print(f"5-FOLD CROSS-VALIDATION RESULTS")
    print(f"{'='*60}")
    print(f"Mean Accuracy: {avg_acc*100:.2f}% ± {std_acc*100:.2f}%")
    print(f"Average Epochs: {avg_epochs:.1f}")
    print(f"\nIndividual Fold Results:")
    for r in fold_results:
        print(f"  Fold {r['fold']}: {r['best_val_acc']*100:.2f}% (epoch {r['stopped_epoch']})")
    print(f"{'='*60}")


RESUMING TRAINING FROM CHECKPOINT

Fold results collected so far: 3/5
Completed folds: 3
Next fold to train: 4

CONTINUING TRAINING FROM FOLD 4

FOLD 4/5
Created SGD optimizer: lr=0.001, momentum=0.8, weight_decay=0.0001
Training on: cpu

Epoch 1/30
Created SGD optimizer: lr=0.001, momentum=0.8, weight_decay=0.0001
Training on: cpu

Epoch 1/30


Evaluating: 100%|██████████| 62/62 [06:13<00:00,  6.02s/it, loss=1.1813, acc=50.60%]



Learning Rate: 0.001000

Epoch 1 Results:
   Train Loss: 1.2920 | Train Acc: 41.00%
   Val Loss:   1.1974 | Val Acc:   50.60%
SUCCESS: Checkpoint saved: checkpoints/resnet101_fold4/fold_1/checkpoint_epoch_1.pth
SUCCESS: Best model saved: checkpoints/resnet101_fold4/fold_1/best_accuracy.pt
New best model! Val Acc: 50.60%

Epoch 2/30
SUCCESS: Checkpoint saved: checkpoints/resnet101_fold4/fold_1/checkpoint_epoch_1.pth
SUCCESS: Best model saved: checkpoints/resnet101_fold4/fold_1/best_accuracy.pt
New best model! Val Acc: 50.60%

Epoch 2/30


Evaluating: 100%|██████████| 62/62 [07:01<00:00,  6.79s/it, loss=0.9838, acc=60.69%]



Learning Rate: 0.001000

Epoch 2 Results:
   Train Loss: 1.0708 | Train Acc: 56.39%
   Val Loss:   0.9895 | Val Acc:   60.69%
SUCCESS: Checkpoint saved: checkpoints/resnet101_fold4/fold_1/checkpoint_epoch_2.pth
SUCCESS: Best model saved: checkpoints/resnet101_fold4/fold_1/best_accuracy.pt
New best model! Val Acc: 60.69%

Epoch 3/30
SUCCESS: Checkpoint saved: checkpoints/resnet101_fold4/fold_1/checkpoint_epoch_2.pth
SUCCESS: Best model saved: checkpoints/resnet101_fold4/fold_1/best_accuracy.pt
New best model! Val Acc: 60.69%

Epoch 3/30


Evaluating: 100%|██████████| 62/62 [06:45<00:00,  6.54s/it, loss=0.8641, acc=66.48%]



Learning Rate: 0.001000

Epoch 3 Results:
   Train Loss: 0.9031 | Train Acc: 62.45%
   Val Loss:   0.8418 | Val Acc:   66.48%
SUCCESS: Checkpoint saved: checkpoints/resnet101_fold4/fold_1/checkpoint_epoch_3.pth
SUCCESS: Best model saved: checkpoints/resnet101_fold4/fold_1/best_accuracy.pt
New best model! Val Acc: 66.48%

Epoch 4/30
SUCCESS: Checkpoint saved: checkpoints/resnet101_fold4/fold_1/checkpoint_epoch_3.pth
SUCCESS: Best model saved: checkpoints/resnet101_fold4/fold_1/best_accuracy.pt
New best model! Val Acc: 66.48%

Epoch 4/30


Evaluating: 100%|██████████| 62/62 [07:11<00:00,  6.96s/it, loss=0.8266, acc=72.58%]



Learning Rate: 0.001000

Epoch 4 Results:
   Train Loss: 0.8036 | Train Acc: 67.75%
   Val Loss:   0.7378 | Val Acc:   72.58%
SUCCESS: Checkpoint saved: checkpoints/resnet101_fold4/fold_1/checkpoint_epoch_4.pth
SUCCESS: Best model saved: checkpoints/resnet101_fold4/fold_1/best_accuracy.pt
New best model! Val Acc: 72.58%

Epoch 5/30
SUCCESS: Checkpoint saved: checkpoints/resnet101_fold4/fold_1/checkpoint_epoch_4.pth
SUCCESS: Best model saved: checkpoints/resnet101_fold4/fold_1/best_accuracy.pt
New best model! Val Acc: 72.58%

Epoch 5/30


Evaluating:  87%|████████▋ | 54/62 [05:37<00:51,  6.41s/it, loss=0.7268, acc=72.40%]

## 5. Cross-Validation Results & Visualizations

Generate comprehensive visualizations and save cross-validation results:
- Confusion matrices (best fold + aggregated across all folds)
- ROC curves for each class
- Training history plots (loss and accuracy across all folds)
- JSON results file for model comparison

In [ ]:
# Configuration
MODEL_NAME = 'ResNet101'
BATCH_SIZE = 32
RESULTS_DIR = 'results'
CHECKPOINT_DIR = 'checkpoints'
N_FOLDS = 5
DPI = 300

# Plotting configuration
PLOT_CONFIG = {
    'cm': {'size': (10, 8), 'cmap': 'Blues'},
    'roc': {'size': (10, 8), 'lw': 2},
    'history': {'size': (18, 10)},
    'comparison': {'size': (10, 6)}
}
FONT = {'title': 14, 'label': 12, 'legend': 10, 'tick': 10}

# Save Cross-Validation Results and Generate Plots
import json
import os
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
import seaborn as sns

# Create model-specific results directory
model_results_dir = os.path.join(RESULTS_DIR, MODEL_NAME.lower())
os.makedirs(model_results_dir, exist_ok=True)
print(f"✓ Results directory created: {model_results_dir}")

# Verify directory exists
if os.path.exists(model_results_dir):
    print(f"✓ Directory verified: {os.path.abspath(model_results_dir)}")
else:
    print(f"⚠ Warning: Directory not found, creating again...")
    os.makedirs(model_results_dir, exist_ok=True)

# Get best fold
best_fold_idx = np.argmax([r['best_val_acc'] for r in fold_results])
best_fold_num = fold_results[best_fold_idx]['fold']

# Recreate validation set for best fold
fold_splits = list(kfold.split(X_all, y_all))
train_idx, val_idx = fold_splits[best_fold_idx]
val_dataset = DFUDataset(X_all[val_idx], y_all[val_idx], transform=val_test_transform)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Load best model
checkpoint_path = f'{CHECKPOINT_DIR}/resnet101_fold{best_fold_num}/fold_1/best_accuracy.pt'

# Check if checkpoint exists
if not os.path.exists(checkpoint_path):
    print(f"Warning: Checkpoint not found at {checkpoint_path}")

else:
    model = create_resnet101_model(num_classes=num_classes, pretrained=False)
    model.load_state_dict(torch.load(checkpoint_path))
    model = model.to(device).eval()
    print(f"✓ Loaded best model from {checkpoint_path}")

# Get predictions
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        outputs = model(inputs.to(device))
        probs = torch.softmax(outputs, dim=1)
        all_probs.append(probs.cpu().numpy())
        all_preds.append(torch.max(outputs, 1)[1].cpu().numpy())
        all_labels.append(labels.numpy())

predictions = np.concatenate(all_preds)
true_labels = np.concatenate(all_labels)
y_pred_proba = np.vstack(all_probs)

# 1. Confusion Matrix (Raw Counts)
cm = confusion_matrix(true_labels, predictions)

fig, ax = plt.subplots(figsize=PLOT_CONFIG['cm']['size'])
sns.heatmap(cm, annot=True, fmt='d', cmap=PLOT_CONFIG['cm']['cmap'],
            xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_xlabel('Predicted', fontsize=FONT['label'], fontweight='bold')
ax.set_ylabel('True', fontsize=FONT['label'], fontweight='bold')
ax.set_title(f'{MODEL_NAME} - Confusion Matrix (Fold {best_fold_num})', 
             fontsize=FONT['title'], fontweight='bold')
plt.tight_layout()
os.makedirs(model_results_dir, exist_ok=True)  # Ensure directory exists
plt.savefig(f'{model_results_dir}/{MODEL_NAME.lower()}_confusion_matrix.png', dpi=DPI, bbox_inches='tight')
plt.show()

# 2. Aggregated Confusion Matrix (All 5 Folds)
print("\n⏳ Generating aggregated confusion matrix across all folds...")
cm_aggregated = np.zeros((num_classes, num_classes), dtype=int)

for fold_idx, (train_idx, val_idx) in enumerate(fold_splits, 1):
    # Recreate validation set for this fold
    val_dataset_fold = DFUDataset(X_all[val_idx], y_all[val_idx], transform=val_test_transform)
    val_loader_fold = DataLoader(val_dataset_fold, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # Load model for this fold
    fold_checkpoint_path = f'{CHECKPOINT_DIR}/resnet101_fold{fold_idx}/fold_1/best_accuracy.pt'
    if os.path.exists(fold_checkpoint_path):
        model_fold = create_resnet101_model(num_classes=num_classes, pretrained=False)
        model_fold.load_state_dict(torch.load(fold_checkpoint_path))
        model_fold = model_fold.to(device).eval()
        
        # Get predictions for this fold
        fold_preds, fold_labels = [], []
        with torch.no_grad():
            for inputs, labels in val_loader_fold:
                outputs = model_fold(inputs.to(device))
                fold_preds.append(torch.max(outputs, 1)[1].cpu().numpy())
                fold_labels.append(labels.numpy())
        
        # Accumulate confusion matrix
        fold_predictions = np.concatenate(fold_preds)
        fold_true_labels = np.concatenate(fold_labels)
        cm_fold = confusion_matrix(fold_true_labels, fold_predictions, labels=range(num_classes))
        cm_aggregated += cm_fold
        print(f"  ✓ Fold {fold_idx} processed")
    else:
        print(f"  ⚠ Fold {fold_idx} checkpoint not found, skipping")

# Normalize aggregated confusion matrix
cm_aggregated_norm = cm_aggregated.astype('float') / cm_aggregated.sum(axis=1)[:, np.newaxis]

# Plot aggregated confusion matrix
fig, ax = plt.subplots(figsize=PLOT_CONFIG['cm']['size'])
sns.heatmap(cm_aggregated_norm, annot=True, fmt='.2%', cmap=PLOT_CONFIG['cm']['cmap'],
            xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_xlabel('Predicted', fontsize=FONT['label'], fontweight='bold')
ax.set_ylabel('True', fontsize=FONT['label'], fontweight='bold')
ax.set_title(f'{MODEL_NAME} - Aggregated Confusion Matrix (5-Fold CV)', 
             fontsize=FONT['title'], fontweight='bold')
plt.tight_layout()
os.makedirs(model_results_dir, exist_ok=True)  # Ensure directory exists
plt.savefig(f'{model_results_dir}/{MODEL_NAME.lower()}_confusion_matrix_aggregated.png', dpi=DPI, bbox_inches='tight')
plt.show()

print(f"✓ Aggregated confusion matrix saved ({cm_aggregated.sum()} total predictions)")

# 3. ROC Curve
y_true_bin = label_binarize(true_labels, classes=range(num_classes))
fig, ax = plt.subplots(figsize=PLOT_CONFIG['roc']['size'])

for i in range(num_classes):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
    ax.plot(fpr, tpr, linewidth=PLOT_CONFIG['roc']['lw'], 
            label=f'{classes[i]} (AUC={auc(fpr, tpr):.2f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('FPR', fontsize=FONT['label'], fontweight='bold')
ax.set_ylabel('TPR', fontsize=FONT['label'], fontweight='bold')
ax.set_title(f'{MODEL_NAME} - ROC Curve (Fold {best_fold_num})', 
             fontsize=FONT['title'], fontweight='bold')
ax.legend(loc='lower right', fontsize=FONT['legend'])
ax.grid(alpha=0.3)
plt.tight_layout()
os.makedirs(model_results_dir, exist_ok=True)  # Ensure directory exists
plt.savefig(f'{model_results_dir}/{MODEL_NAME.lower()}_roc_curve.png', dpi=DPI, bbox_inches='tight')
plt.show()

# 4. All Folds Training History (Loss)
fig, axes = plt.subplots(2, 3, figsize=PLOT_CONFIG['history']['size'])
fig.suptitle(f'{MODEL_NAME} - {N_FOLDS}-Fold CV (Loss)', fontsize=FONT['title']+2, fontweight='bold')

for idx, r in enumerate(fold_results):
    ax = axes[idx // 3, idx % 3]
    history = r['history']
    epochs = range(1, len(history['train_loss']) + 1)
    
    ax.plot(epochs, history['train_loss'], 'b-', label='Train', lw=2)
    ax.plot(epochs, history['val_loss'], 'r-', label='Val', lw=2)
    ax.axvline(x=history['val_loss'].index(min(history['val_loss'])) + 1, 
               color='g', linestyle='--', alpha=0.5, label='Best Val')
    if r['stopped_epoch'] < len(epochs):
        ax.axvline(x=r['stopped_epoch'], color='orange', linestyle='--', alpha=0.5, label='Stopped')
    
    ax.set_xlabel('Epoch', fontsize=FONT['tick'])
    ax.set_ylabel('Loss', fontsize=FONT['tick'])
    ax.set_title(f"Fold {r['fold']}: {r['best_val_acc']*100:.2f}%", fontweight='bold')
    ax.legend(fontsize=FONT['tick']-2)
    ax.grid(alpha=0.3)

axes[1, 2].axis('off')
plt.tight_layout()
os.makedirs(model_results_dir, exist_ok=True)  # Ensure directory exists
plt.savefig(f'{model_results_dir}/{MODEL_NAME.lower()}_all_folds_loss.png', dpi=DPI, bbox_inches='tight')
plt.show()

# 5. All Folds Training History (Accuracy)
fig, axes = plt.subplots(2, 3, figsize=PLOT_CONFIG['history']['size'])
fig.suptitle(f'{MODEL_NAME} - {N_FOLDS}-Fold CV (Accuracy)', fontsize=FONT['title']+2, fontweight='bold')

for idx, r in enumerate(fold_results):
    ax = axes[idx // 3, idx % 3]
    history = r['history']
    epochs = range(1, len(history['train_acc']) + 1)
    
    # Convert to percentage
    train_acc_pct = [acc * 100 for acc in history['train_acc']]
    val_acc_pct = [acc * 100 for acc in history['val_acc']]
    
    ax.plot(epochs, train_acc_pct, 'b-', label='Train', lw=2)
    ax.plot(epochs, val_acc_pct, 'r-', label='Val', lw=2)
    ax.axvline(x=history['val_acc'].index(max(history['val_acc'])) + 1, 
               color='g', linestyle='--', alpha=0.5, label='Best Val')
    if r['stopped_epoch'] < len(epochs):
        ax.axvline(x=r['stopped_epoch'], color='orange', linestyle='--', alpha=0.5, label='Stopped')
    
    ax.set_xlabel('Epoch', fontsize=FONT['tick'])
    ax.set_ylabel('Accuracy (%)', fontsize=FONT['tick'])
    ax.set_title(f"Fold {r['fold']}: {r['best_val_acc']*100:.2f}%", fontweight='bold')
    ax.legend(fontsize=FONT['tick']-2)
    ax.grid(alpha=0.3)
    ax.set_ylim([0, 100])

axes[1, 2].axis('off')
plt.tight_layout()
os.makedirs(model_results_dir, exist_ok=True)  # Ensure directory exists
plt.savefig(f'{model_results_dir}/{MODEL_NAME.lower()}_all_folds_accuracy.png', dpi=DPI, bbox_inches='tight')
plt.show()

# Save results
results = {
    'model_name': MODEL_NAME,
    'cv_results': {
        'val_accuracy': {'mean': float(avg_acc), 'std': float(std_acc)},
        'avg_epochs': float(avg_epochs),
        'fold_results': [{'fold': r['fold'], 'best_val_acc': float(r['best_val_acc']), 
                          'stopped_epoch': int(r['stopped_epoch'])} for r in fold_results]
    },
    'best_fold': {
        'fold_number': best_fold_num, 
        'best_val_acc': float(fold_results[best_fold_idx]['best_val_acc']),
        'checkpoint_path': checkpoint_path
    }
}

os.makedirs(model_results_dir, exist_ok=True)  # Ensure directory exists
with open(f'{model_results_dir}/{MODEL_NAME.lower()}_cv_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n{'='*60}")
print(f"{MODEL_NAME} CROSS-VALIDATION SUMMARY")
print(f"{'='*60}")
print(f"Mean Accuracy: {avg_acc*100:.2f}% ± {std_acc*100:.2f}%")
print(f"Best Fold: {best_fold_num} ({fold_results[best_fold_idx]['best_val_acc']*100:.2f}%)")
print(f"Average Epochs: {avg_epochs:.1f}")
print(f"\nResults saved to: {model_results_dir}/{MODEL_NAME.lower()}_cv_results.json")
print(f"{'='*60}")